# 01 — Data Exploration

Fetch BTC/USD daily OHLCV data, compute technical indicators, and save to parquet.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

from hmats.data.pipeline import compute_indicators

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-darkgrid")
pd.options.display.max_columns = 30

## 1. Fetch OHLCV data

In [ ]:
TICKER = "BTC-USD"
START = "2020-01-01"

raw = yf.download(TICKER, start=START, interval="1d", auto_adjust=True)

# yfinance may return MultiIndex columns for a single ticker — flatten
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

print(f"Shape: {raw.shape}")
raw.tail()

## 2. Compute technical indicators

In [ ]:
df = compute_indicators(raw.copy())
df.tail()

## 3. Visualise

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Price + SMAs + Bollinger Bands
ax = axes[0]
ax.plot(df.index, df["Close"], label="Close", linewidth=1)
for sma in ["SMA_20", "SMA_50", "SMA_200"]:
    ax.plot(df.index, df[sma], label=sma, linewidth=0.8, linestyle="--")
ax.fill_between(
    df.index, df["BB_Upper"], df["BB_Lower"], alpha=0.1, color="grey", label="Bollinger Bands"
)
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"{TICKER} — Price & Moving Averages")

# RSI
ax = axes[1]
ax.plot(df.index, df["RSI_14"], color="purple", linewidth=0.9)
ax.axhline(70, color="red", linestyle="--", linewidth=0.7)
ax.axhline(30, color="green", linestyle="--", linewidth=0.7)
ax.set_ylabel("RSI (14)")
ax.set_title("RSI")

# MACD
ax = axes[2]
ax.plot(df.index, df["MACD"], label="MACD", linewidth=0.9)
ax.plot(df.index, df["MACD_Signal"], label="Signal", linewidth=0.9)
ax.bar(df.index, df["MACD_Hist"], label="Histogram", alpha=0.4, width=1)
ax.set_ylabel("MACD")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("MACD")

# Volume
ax = axes[3]
ax.bar(df.index, df["Volume"], alpha=0.6, width=1)
ax.set_ylabel("Volume")
ax.set_title("Volume")

fig.tight_layout()
plt.show()

## 4. Save to parquet

In [ ]:
out_dir = Path.cwd().parents[2] / "data" / "raw"
# Fallback: if running from the repo root
if not out_dir.exists():
    out_dir = Path("data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "btc_daily.parquet"
df.to_parquet(out_path)
print(f"Saved {len(df)} rows → {out_path.resolve()}")

In [ ]:
# Quick sanity check: reload and inspect
reloaded = pd.read_parquet(out_path)
print(f"Columns: {list(reloaded.columns)}")
print(f"Date range: {reloaded.index.min()} → {reloaded.index.max()}")
reloaded.describe().round(2)